In [53]:
# Load with MultiIndex headers (two rows of headers)
# When loading the pickle, check shape and head immediately
with open('../data/processed/seera_cleaned.pkl', 'rb') as f:
    df_clean = pickle.load(f)

print("Shape:", df_clean.shape)
print("Column levels:", df_clean.columns.nlevels)
print("Head:\n", df_clean.head(2))

print("Dataset loaded successfully!")
print(f"Shape: {df_clean.shape} rows x {df_clean.shape[1]} columns")

Shape: (119, 52)
Column levels: 2
Head:
   General Information                                       \
      Year of project Organization id Size of organization   
0                2015               1                   16   
1                2016              25                    2   

                                                              \
  Size of IT department Estimated  duration Development type   
0                     7                 2.0                1   
1                     1                 2.0                1   

                              Size        Effort       Environment  ...  \
  Application domain Object points Actual effort Contract maturity  ...   
0                  7         100.0        3168.0                 8  ...   
1                  3          19.0        1584.0                 4  ...   

                    Project                                         \
  Degree of risk management Use of standards Process reengineering   
0           

<h2>Spliting the dataset </h2>

In [ ]:
df_clean = df_clean.sort_index(axis=1)
x = df_clean.drop(columns=['Effort_Actual effort', 'General Information_Organization id', 'General Information_ProjID'])
y = np.log1p(df_clean['Effort_Actual effort'])

In [35]:
categorical_cols = [

('Developers_Team selection'),	
('Developers_DBMS expert availability'),
('General Information_Development type'),
('General Indformation_Application domain'),
('General Information_Size of organization'),
('General_information_Size of IT department'),
('General Information_Organization type'),
('Environment_Government policy impact'),
('Developers_Developer hiring policy'),
('Developers_Developer training')
]

In [ ]:
# Print all column names as a list
print(df_clean.columns.tolist())

In [ ]:
missing_cat_cols = [c for c in categorical_cols if c not in X.columns]

print("Categorical columns:", len(categorical_cols))
print("Not found:", missing_cat_cols)

In [ ]:
# Ensure groups is a 1D pandas Series
groups = df_clean['General Information_Organization id']

# If df_clean has duplicate column names, select just the first one:
if isinstance(groups, pd.DataFrame):
    groups = groups.iloc[:, 0]

logo = LeaveOneGroupOut()

for fold, (train_idx, test_idx) in enumerate(logo.split(x, y, groups=groups)):
    X_train = x.iloc[train_idx]
    X_test  = x.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]
    #Imputation: but the strategy is different depending on the type of data. For numerical data, we can use mean or median, while for categorical data, we can use the most frequent value.
    imputer.fit(X_train)
    X_train = imputer.transform(X_train)
    X_test = imputer.transform(X_test)


In [ ]:
print("Numerical columns:")
print(X.select_dtypes(include='number').columns.tolist())

print("\nCategorical columns:")
print(X.select_dtypes(exclude='number').columns.tolist())

print("\nMissing values:")
print(X.isna().sum()[X.isna().sum() > 0].sort_values(ascending=False))

In [ ]:
print("Missing target:",
      y.isna().sum())

print("Target rows:",
      len(y))

In [45]:

# ==========================================
# 1. SANITIZE COLUMN NAMES & DEDUPLICATE
# ==========================================
if isinstance(df_clean.columns, pd.MultiIndex):
    df_clean.columns = ['_'.join(map(str, col)).strip() for col in df_clean.columns]

df_clean.columns = (
    df_clean.columns
    .astype(str)
    .str.replace('\xa0', ' ', regex=False)
    .str.replace('\u200b', '', regex=False)  # zero-width space
    .str.strip()
)

df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()].copy()

# DEBUG — print all columns to find exact name
print("All columns after cleaning:")
for col in df_clean.columns:
    print(repr(col))  # repr shows hidden characters

All columns after cleaning:
'Developers_Analysts capability_1.0'
'Developers_Consultant availability_3.0'
'Developers_DBMS  expert availability_0'
'Developers_Daily working hours_8'
'Developers_Dedicated team members_6'
'Developers_Income satisfaction_3.0'
'Developers_Precedentedness_2.0'
'Developers_Programmers capability_2.0'
'Developers_Programmers experience in programming language_2.0'
'Developers_Project manager experience_2.0'
'Developers_Software tool experience_2.0'
'Developers_Team cohesion_2.5'
'Developers_Team contracts_1.0'
'Developers_Team selection_3.0'
'Developers_Team size_6'
'Effort_Actual effort_3168.0'
'Environment_Contract maturity_8'
'Environment_Developer hiring policy_1.0'
'Environment_Developer incentives policy_8.0'
'Environment_Developer training_3.0'
'Environment_Development team management_4.0'
'Environment_Economic instability impact_1'
'Environment_Government policy impact_4.0'
'Environment_Organization management structure clarity_2.0'
'General Informati